# Hive 商圈聚类任务

In [ ]:
import sys
from tempfile import NamedTemporaryFile
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import spdbccc_data as sd
from spdbccc_data import dtDate
from spdbccc_data import task as taskfinish
from business_district.hive_task import (
    HiveTaskConfig as InitialHiveTaskConfig,
    TaskMain as InitialTaskMain,
    load_hive_algorithm_parameters,
)
from business_district.resource_usage import (
    capture_resource_usage,
    print_resource_usage,
    start_resource_tracking,
    stop_resource_tracking,
)
from incremental_assignment.hive_task import (
    DEFAULT_ASSIGNMENT_CONFIG,
    DEFAULT_GRAPH_CONFIG,
    DEFAULT_TIMESTAMP_FORMATS,
    DEFAULT_VISIT_CONFIG,
    HiveTaskConfig as IncrementalHiveTaskConfig,
    TaskMain as IncrementalTaskMain,
)

In [ ]:
INLINE_SHANGHAI_CONFIG = f"""
[city]
code = "shanghai"
name = "上海市"

[input]
transactions_path = "../data.txt"
timestamp_formats = %Y%m%dT%H%M%S, %Y%m%d%H%M%S, %Y-%m-%d %H:%M:%S, %Y/%m/%d %H:%M:%S

[visits]
merge_window_minutes = 20
maximum_daily_merchants_per_card = 20

[cooccurrence]

[graph]
edge_weight_method = "sppmi"
context_smoothing_alpha = 0.75
sppmi_shift = 3.0
top_k_neighbors = 8
minimum_z_score = 0.5
degree_penalty_gamma = 0.5
jaccard_threshold = 0.1

[community]
algorithm = "leiden"
resolution = 1.0
random_seed = 42
maximum_cleaning_rounds = 3
minimum_hub_degree = 6
participation_threshold = 0.70

[geo]
cluster_radius_meters = 1000.0

[anchors]
minimum_count = 3
maximum_count = 10
merchants_per_anchor = 20
maximum_participation = 0.1
chain_visit_count_quantile = 0.9
chain_minimum_visit_count = 100

[output]
directory = "../algorithm_one_output"
""".strip()

with NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".ini", delete=False) as config_file:
    config_file.write(INLINE_SHANGHAI_CONFIG)
    config_file.write("\n")
    inline_config_path = Path(config_file.name)

inline_config_path

In [ ]:
initial_task_config = InitialHiveTaskConfig(
    config_path=inline_config_path,
    source_table="dev_icamp.icamp_merchant_cluster_algo_input",
    parameter_table="dev_icamp.icamp_merchant_cluster_algo_param",
    target_table="dev_icamp.icamp_merchant_cluster_algo_output",
    target_temp_table="dev_icamp.icamp_merchant_cluster_algo_output_tmp",
    dt_expression="T-1",
)

incremental_task_config = IncrementalHiveTaskConfig(
    config_path=inline_config_path,
    timestamp_formats=DEFAULT_TIMESTAMP_FORMATS,
    visit_config=DEFAULT_VISIT_CONFIG,
    graph_config=DEFAULT_GRAPH_CONFIG,
    assignment_config=DEFAULT_ASSIGNMENT_CONFIG,
    source_table="dev_icamp.icamp_merchant_cluster_algo_input",
    parameter_table="dev_icamp.icamp_merchant_cluster_algo_param",
    target_table="dev_icamp.icamp_merchant_cluster_algo_output",
    target_temp_table="dev_icamp.icamp_merchant_cluster_algo_output_incremental_tmp",
    dt_expression="T-1",
)

initial_task_config, incremental_task_config

In [ ]:
parameter_dt = str(dtDate.dt_date(initial_task_config.dt_expression))
parameter_data = sd.read_table(
    initial_task_config.parameter_table,
    dt=[parameter_dt],
)
parameters = load_hive_algorithm_parameters(
    parameter_data,
    initial_task_config.parameter_table,
)
if len(parameters) != 1:
    raise ValueError(
        "参数表 T-1 分区必须只有一条参数记录: "
        f"table={initial_task_config.parameter_table}, "
        f"dt={parameter_dt}, row_count={len(parameters)}"
    )

is_daily = parameters[0].is_daily
task_config = incremental_task_config if is_daily else initial_task_config
task_class = IncrementalTaskMain if is_daily else InitialTaskMain
task_mode = "incremental_assignment" if is_daily else "initial_clustering"
print(f"task_mode={task_mode}, parameter_dt={parameter_dt}, is_daily={is_daily}")

resource_started_at = start_resource_tracking()
task = task_class(task_config)
try:
    task.check()
    summary = task.taskrun()
finally:
    task.destroy()
    resource_usage = capture_resource_usage(resource_started_at)
    stop_resource_tracking()
    taskfinish.finish_task()

print_resource_usage(resource_usage)

summary